# CodeRL — base-model evaluation on Kaggle (Phase 2a)

Measures **Qwen2.5-Coder-0.5B-Instruct** and **Qwen2.5-Coder-1.5B-Instruct** (no training) on
**HumanEval** (164) and **MBPP-test** (500): greedy pass@1, and unbiased pass@1 / pass@10 from
n=20 samples at T=0.8, top_p=0.95. Every completion runs in the CodeRL sandbox (`eval` limit
profile).

### Kaggle settings (right-hand panel → *Settings*) — set these BEFORE running
| Setting | Value |
|---|---|
| **Accelerator** | **GPU T4 x2** |
| **Internet** | **On** (needs a phone-verified Kaggle account) |
| **Persistence** | **Files only** (helps retain checkpoints between interactive sessions; also download results) |
| **Secrets** (*Add-ons → Secrets*) | `WANDB_API_KEY` (optional: logs to W&B project `coderl`), `HF_TOKEN` (optional: avoids Hub rate limits). Tick *Attach to notebook* for each. |

Start with **LIMIT = 10** for a dry run. After checking it, set **LIMIT = None** for the full
baseline (164 HumanEval + 500 MBPP-test problems per model). **GENERATION_BACKEND = "hf"**
uses the Hugging Face backend validated by the saved T4 dry run; `"auto"` tries vLLM first.
Dry runs write to `results/kaggle_dry_run/`; full runs write to `results/base/`.
Run **one model at a time**: `MODEL_TAG="qwen0.5b"` first, then download its archive before
changing to `"qwen1.5b"`. The previous full run exhausted T4 memory with 80 HF sequences;
this notebook uses **HF_BATCH_SIZE=20**, **EVAL_BATCH_SIZE=8**, and saves completed batches.
For a long unattended run, choose **Save Version → Save & Run All**; it starts a separate,
clean session. It does not resume files from the interactive session. For an interactive
retry with retained files, rerun with the same settings; `--resume` reuses completed batches.

Checkpoints need the files to survive. They cannot recover an already deleted runtime.
Download the archive after each model and keep **Files only** persistence enabled.

Then **Run All**. With vLLM, expected time: ~10 min install, then roughly 30–90 min of evaluation
(time depends on the selected model). If vLLM cannot be installed, it falls back to Hugging
Face `transformers` automatically (much slower: several hours). Troubleshooting:
[`docs/KAGGLE.md`](https://github.com/Varad2812/CodeRL/blob/main/docs/KAGGLE.md).

**Output:** `/kaggle/working/coderl_results_base_<MODEL_TAG>.zip` for a full selected-model run;
dry runs use `coderl_results_dry_run_<MODEL_TAG>.zip` (*Output* tab → download). Unzip it into
the repo root; only the `summary.json` files are committed (samples and logs are gitignored).


## 1. GPU and Python

In [ ]:
import subprocess
import sys

print("Python", sys.version)
smi = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
print(smi.stdout or smi.stderr)
if smi.returncode != 0:
    print("WARNING: no GPU visible. Settings -> Accelerator -> 'GPU T4 x2', then restart.")

## 2. Configuration (edit here if needed)

In [ ]:
REPO_URL = "https://github.com/Varad2812/CodeRL"
REPO_REF = "codex/kaggle-baseline-handoff"  # includes checkpoint/recovery support
WORKDIR = "/kaggle/working/CodeRL"
MODEL_TAG = "qwen0.5b"                  # finish/download this model, then use "qwen1.5b"
ALL_MODELS = {                         # results/base/<tag>/
    "qwen0.5b": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "qwen1.5b": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
}
MODELS = {MODEL_TAG: ALL_MODELS[MODEL_TAG]}
BENCHMARKS = "humaneval,mbpp_test"
LIMIT = 10                             # None for the FULL baseline; 10 for a dry run
GENERATION_BACKEND = "hf"              # "hf" (validated on T4) or "auto" (try vLLM first)
N_SAMPLES = 20
DTYPE = "float16"                      # T4 has no bfloat16
GPU_MEMORY_UTILIZATION = 0.85          # vLLM only: lower (0.7) on OOM
MAX_MODEL_LEN = 2048                   # vLLM only
HF_BATCH_SIZE = 20                     # HF: 20 sequences, rather than the failing 80
EVAL_BATCH_SIZE = 8                    # checkpoint and report every 8 problems

if LIMIT is not None and (type(LIMIT) is not int or LIMIT <= 0):
    raise ValueError("LIMIT must be None (full baseline) or a positive integer (dry run)")
if GENERATION_BACKEND not in ("hf", "auto"):
    raise ValueError('GENERATION_BACKEND must be "hf" or "auto"')
RESULTS_DIR = "results/kaggle_dry_run" if LIMIT is not None else "results/base"
ARCHIVE_NAME = ("coderl_results_dry_run" if LIMIT is not None else "coderl_results_base") + "_" + MODEL_TAG
print("Run:", "DRY RUN" if LIMIT is not None else "FULL BASELINE", "| output:", RESULTS_DIR)

## 3. Clone the repo and install
With `GENERATION_BACKEND="hf"`, installs only the `hf` extra and skips vLLM.
With `"auto"`, installs CodeRL with the `gen` extra (pins `vllm==0.18.1`, which replaces Kaggle's torch with
torch 2.10). vLLM is then imported in a *fresh* interpreter; if that fails, the run uses the
Hugging Face backend instead (printed loudly below).

In [ ]:
import os
import subprocess
import sys


def sh(cmd, check=True):
    print("$", cmd, flush=True)
    rc = subprocess.run(cmd, shell=True).returncode
    if check and rc != 0:
        raise RuntimeError(f"command failed ({rc}): {cmd}")
    return rc


if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    sh(f"git clone --depth 1 --branch {REPO_REF} {REPO_URL} {WORKDIR}")
else:
    sh(f"git -C {WORKDIR} fetch --depth 1 origin {REPO_REF} && git -C {WORKDIR} checkout -q FETCH_HEAD")
os.chdir(WORKDIR)
sh("git log --oneline -1")

PY = sys.executable
if GENERATION_BACKEND == "hf":
    sh(f'{PY} -m pip install -q -e ".[hf]"')
    BACKEND = "hf"
    print("Hugging Face selected explicitly; skipping vLLM install and engine startup.")
else:
    if sh(f'{PY} -m pip install -q -e ".[gen]"', check=False) != 0:
        print("\n*** WARNING: installing the [gen] extra (vLLM) failed; installing [hf] only ***\n")
        sh(f'{PY} -m pip install -q -e ".[hf]"')

    probe = subprocess.run(
        [PY, "-c", "import vllm, torch; print(vllm.__version__, torch.__version__, torch.version.cuda, torch.cuda.is_available())"],
        capture_output=True, text=True,
    )
    if probe.returncode == 0 and probe.stdout.strip().endswith("True"):
        BACKEND = "vllm"
        print("vLLM OK (vllm, torch, cuda, gpu):", probe.stdout.strip())
    else:
        BACKEND = "hf"
        print("\n" + "*" * 80)
        print("WARNING: vLLM is not usable here -> falling back to the Hugging Face backend.")
        print("Results are equivalent but generation is MUCH slower (hours). Details:")
        print((probe.stdout + probe.stderr)[-3000:])
        print("*" * 80 + "\n")
print("generation backend:", BACKEND)

## 4. Sandbox isolation
Tries to install `bubblewrap` (non-fatal). The isolation level actually used is also stored
in every `summary.json` (`sandbox.isolation`: `bwrap` > `unshare` > `none`).

In [ ]:
import json

if sh("apt-get install -y -qq bubblewrap > /dev/null 2>&1", check=False) != 0:
    sh("(apt-get update -qq && apt-get install -y -qq bubblewrap) > /dev/null 2>&1", check=False)

info = subprocess.run([PY, "-m", "coderl.sandbox", "info"], capture_output=True, text=True)
print(info.stdout or info.stderr)
try:
    ISOLATION = json.loads(info.stdout[info.stdout.index("{"):])["subprocess_isolation"]
except (ValueError, KeyError):
    ISOLATION = "unknown"
print("sandbox isolation:", ISOLATION)
if ISOLATION != "bwrap":
    print("NOTE: weaker sandbox isolation than bwrap (common inside Kaggle's container). OK for "
          "evaluating base models: the eval processes below get no secrets, and the level is "
          "recorded in every summary.json.")

## 5. Secrets (optional)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient

    _secrets = UserSecretsClient()
    for _name in ("WANDB_API_KEY", "HF_TOKEN"):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
            print(f"{_name}: loaded")
        except Exception:
            print(f"{_name}: not set (optional)")
except Exception as exc:
    print("Kaggle secrets unavailable (not on Kaggle?):", repr(exc))

## 6. Build the datasets and pre-download the models (HF Hub)

In [ ]:
sh(f"{PY} -m coderl.data.build --out data/processed")
sh("wc -l data/processed/*.jsonl")
# Download weights here, in the notebook kernel (which holds HF_TOKEN), so the evaluation
# processes below can run WITHOUT any secrets in their environment.
for _model in MODELS.values():
    sh(f"{PY} -c \"from huggingface_hub import snapshot_download as s; print(s('{_model}'))\"")

## 7. Evaluate both base models
The selected model runs in a separate process on GPU 0. Download its results before
selecting the other model in cell 2. Progress (JSON lines) is printed every minute; full
logs are in `<RESULTS_DIR>/logs/`.

The evaluation processes get **no secrets** (`WANDB_API_KEY` / `HF_TOKEN` are removed from their
environment): unless the sandbox reached `bwrap` isolation, generated code could read its
parent's environment. W&B logging happens afterwards, in step 8.

In [ ]:
import time

gpus = [l for l in subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.splitlines() if l.startswith("GPU")]
N_GPUS = max(1, len(gpus))
SECRETS = ("WANDB_API_KEY", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "KAGGLE_KEY")
os.makedirs(f"{RESULTS_DIR}/logs", exist_ok=True)


def eval_cmd(tag, model):
    cmd = [
        PY, "-m", "coderl.eval.run_benchmarks",
        "--model", model, "--backend", BACKEND, "--benchmarks", BENCHMARKS,
        "--out", f"{RESULTS_DIR}/{tag}", "--dtype", DTYPE, "--n-samples", str(N_SAMPLES),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--batch-size", str(EVAL_BATCH_SIZE), "--hf-batch-size", str(HF_BATCH_SIZE),
        "--resume",
        "--sandbox-workers", "2" if N_GPUS > 1 else "4",
    ]
    return cmd + (["--limit", str(LIMIT)] if LIMIT is not None else [])


def start(tag, model, gpu):
    env = {k: v for k, v in os.environ.items() if k not in SECRETS}
    env.update(CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1")
    log = open(f"{RESULTS_DIR}/logs/{tag}.log", "a")
    print(f"[{tag}] GPU {gpu}: {' '.join(eval_cmd(tag, model))}", flush=True)
    return subprocess.Popen(eval_cmd(tag, model), env=env, stdout=log, stderr=subprocess.STDOUT)


def last_progress(tag):
    try:
        lines = open(f"{RESULTS_DIR}/logs/{tag}.log").read().splitlines()
    except OSError:
        return ""
    keep = [l for l in lines if '"eval_' in l or '"gen_progress"' in l or "WARNING" in l or "generator_fallback" in l]
    return (keep or lines or [""])[-1][:400]


def wait(procs):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(60)
        for tag in procs:
            print(f"[{tag}] {last_progress(tag)}", flush=True)
    failed = [t for t, p in procs.items() if p.returncode != 0]
    for tag in failed:
        print(f"\n===== {tag} FAILED (exit {procs[tag].returncode}); last log lines =====")
        print("\n".join(open(f"{RESULTS_DIR}/logs/{tag}.log").read().splitlines()[-40:]))
    if failed:
        raise RuntimeError(f"evaluation failed for {failed}; see docs/KAGGLE.md troubleshooting")


t0 = time.time()
try:
    for tag, model in MODELS.items():
        wait({tag: start(tag, model, 0)})
    print(f"selected model finished in {(time.time() - t0) / 60:.1f} min")
finally:
    # Run All stops at a failed cell. Still package completed batches for recovery.
    import shutil

    recovery = shutil.make_archive(f"/kaggle/working/{ARCHIVE_NAME}_recovery", "zip",
                                   root_dir=WORKDIR, base_dir=RESULTS_DIR)
    print("Recovery archive (may be incomplete):", recovery)
    print("Download it from Output before ending the session. Checkpoints alone do not "
          "survive deletion of the runtime.")

## 8. Results table (+ W&B upload if `WANDB_API_KEY` was loaded)

In [ ]:
sh(f"{PY} -m coderl.eval.report {RESULTS_DIR} --out {RESULTS_DIR}/RESULTS.md --wandb")

## 9. Package the results for download

In [ ]:
import shutil

archive = shutil.make_archive(f"/kaggle/working/{ARCHIVE_NAME}", "zip", root_dir=WORKDIR, base_dir=RESULTS_DIR)
print("download from the Output tab:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")
print("Dry run only: do not publish baseline scores." if LIMIT is not None else
      "Selected model complete. Download this archive before evaluating the other model. "
      "The full baseline requires 164 HumanEval and 500 MBPP-test problems for BOTH models.")